## Evaluation pipeline for the microlane experiment

In [49]:
# First consider all the variables
# The input image gets resized to a particular level
# Then create a pipeline to feed data into the model
# AFter this process is completed, then process the data
# Then after the processing is done find a way to take output from the model
# Then, convert the output to relevant format, and store it for future use
# Apply relevant computations

In [50]:
# Imports of the Core Packages
import yaml, random
from tqdm.notebook import tqdm

In [51]:
# Import custom libraries located at different folder location + configs
from microlane.utils.load_image import load_image_from_sample, parse_prediction
from microlane.utils.create_settings import create_settings
from microlane.utils.experiment import ExperimentEvaluate
from microlane.utils.loaders import load_dataset, load_model

In [52]:
# First Load the Configuation file
with open("../configs/config.yaml", "r") as file:
    config = yaml.safe_load(file)

### Experiment Settings

In [ ]:
INFERENCE_VIS_NUMBER = 5
SAMPLE_NUMBER = 75
MODEL = "lanenet"
DATASET = "raw"
AUGMENTATION_TYPE = "motion-blur"
EXPERIMENT_NAME = f"{MODEL}_{DATASET}_{AUGMENTATION_TYPE}"
AUGMENTATION_SETTINGS = {}
OUTPUT_LOCATION = "/home/suyog/desktop/projects/microlane/results/EXPERIMENT/raw/lanenet/motion-blur"


### Pre Processing Part

In [54]:
## Load the Corresponding Model and the dataset based on given settings

data  = load_dataset(DATASET, config, SAMPLE_NUMBER)

model = load_model(MODEL, config)


Initializing container on port  8000
/home/suyog/desktop/projects/microlane/microlane/microlane/models/lanenet2/lanenet2
Image 'lanenet2_image:latest' already exists, skipping build.
Container already running: 13e2567a9394


In [55]:
experiment = ExperimentEvaluate(
    
    experiment_name=EXPERIMENT_NAME,
    
    output_dir=OUTPUT_LOCATION
    
)

### Models and Datasets Loaded, Now Processing Part

In [56]:
# Print some basic information of our data

print(f"Total items: {len(data)}\n")

item = data[0]
print(f"Image Path   : {item.image_path}")
print(f"h_samples    : {item.h_samples}")
print(f"lanes        : {item.lanes}")

Total items: 75

Image Path   : /home/suyog/desktop/projects/microlane/results/13-minute-sampling/frame_0000.jpg
h_samples    : [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100, 105, 110, 115, 120, 125, 130, 135, 140, 145, 150, 155, 160, 165, 170, 175, 180, 185, 190, 195, 200, 205, 210, 215, 220, 225, 230, 235, 240, 245, 250, 255, 260, 265, 270, 275, 280, 285, 290, 295, 300, 305, 310, 315, 320, 325, 330, 335, 340, 345, 350, 355, 360, 365, 370, 375, 380, 385, 390, 395, 400, 405, 410, 415, 420, 425, 430, 435, 440, 445, 450, 455, 460, 465, 470, 475, 480, 485, 490, 495, 500, 505, 510, 515, 520, 525, 530, 535, 540, 545, 550, 555, 560, 565, 570, 575, 580, 585, 590, 595, 600, 605, 610, 615, 620, 625, 630, 635, 640, 645, 650, 655, 660, 665, 670, 675, 680, 685, 690, 695, 700, 705, 710, 715, 720, 725, 730, 735, 740, 745, 750, 755, 760, 765, 770, 775, 780, 785, 790, 795, 800, 805, 810, 815, 820, 825, 830, 835, 840, 845, 850, 855, 860, 865, 870, 875, 880, 885, 890

In [57]:
numbers = [random.randint(0, SAMPLE_NUMBER - 1) for _ in range(INFERENCE_VIS_NUMBER)]
print(numbers)

[29, 18, 7, 26, 46]


In [58]:
PRESETS = {
    "normal":       dict(blur=0.0, rotation=0.0, zoom=1.0, lighting=0.0, motion_blur=0.0, shake=False),
    "lighting-d":  dict(blur=0.0, rotation=0.0, zoom=1.0, lighting=-0.4,  motion_blur=0.0, shake=False), 
    "lighting-b":  dict(blur=0.0, rotation=0.0, zoom=1.0, lighting=0.4,  motion_blur=0.0, shake=False),
    "motion-blur":  dict(blur=0.0, rotation=0.0, zoom=1.0, lighting=0.0,  motion_blur=1, shake=False),
    "camera-shake": dict(blur=0, rotation=0.0, zoom=1.0, lighting=0.0,  motion_blur=0, shake=True),
}

AUGMENTATION_SETTINGS = PRESETS[AUGMENTATION_TYPE]


In [59]:
settings = create_settings(
    inference_vis_number=INFERENCE_VIS_NUMBER,
    sample_number=SAMPLE_NUMBER,
    model=MODEL,
    dataset=DATASET,
    augmentation_type=AUGMENTATION_TYPE,
    augmentation_settings=AUGMENTATION_SETTINGS,
    output_path=experiment.folder_dir,
    experiment_name=EXPERIMENT_NAME,
)

### Looping through all the testing examples

In [60]:
for index, testing_example in enumerate(tqdm(data)):

    testing_example.blur        = AUGMENTATION_SETTINGS["blur"]
    testing_example.rotation    = AUGMENTATION_SETTINGS["rotation"]
    testing_example.zoom        = AUGMENTATION_SETTINGS["zoom"]
    testing_example.lighting    = AUGMENTATION_SETTINGS["lighting"]
    testing_example.motion_blur = AUGMENTATION_SETTINGS["motion_blur"]
    
    if AUGMENTATION_SETTINGS['shake']:
        
        testing_example.rotation = random.uniform(-10.0, 10.0)   # degrees
        
        testing_example.zoom     = random.uniform(1, 1.15)  # subtle scale jitter
        

    loaded_image = load_image_from_sample(testing_example)
    
    response = model.predict(loaded_image)
    
    modeloutput = parse_prediction(response)
    
    experiment.store_prediction(modeloutput)
        
    if index in numbers:
        
        experiment.visualize_prediction(modeloutput)
    
    if (index % 100 == 0) and (index != 0):
        
        print(f"Routine Container Restart for Addressing Memory Leak [{int(index/100)}]")
        
        model.container_manager.restart_container()

  0%|          | 0/75 [00:00<?, ?it/s]